In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
-- WITH all_claims AS (
--   SELECT DISTINCT * FROM (
--     SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, KH_PLAN_ID AS plan_id, MEDICAL_EVENT_ID AS claim_id
--     FROM com_edp_prd.com_raw.kom_medical_events
--     UNION
--     SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id, PHARMACY_EVENT_ID AS claim_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     UNION
--     SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, KH_PLAN_ID AS plan_id, MEDICAL_EVENT_ID AS claim_id
--     FROM com_edp_prd.com_raw.kom_medical_events
--   )
-- ),
-- tagging_zip AS (
--   SELECT a.*, b.PROVIDER_ZIP AS hcp_zip
--   FROM all_claims a
--   LEFT JOIN com_raw.kom_providers b
--     ON a.npi = b.NPI AND b.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),
-- tagging_territory AS (
--   SELECT a.*, COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id, COALESCE(b.territory_name, 'Unknown') AS territory
--   FROM tagging_zip a
--   LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
--     ON TRY_CAST(a.hcp_zip AS STRING) = TRY_CAST(b.zipcode AS STRING)
-- ),
-- tagging_payer AS (
--   SELECT a.*, COALESCE(TRY_CAST(b.PAYER_ID AS STRING), 'Unknown') AS payer_id, COALESCE(TRY_CAST(b.PAYER_NAME AS STRING), 'Unknown') AS payer_name
--   FROM tagging_territory a
--   LEFT JOIN com_raw.kom_plans b
--     ON a.plan_id = b.KH_PLAN_ID
-- ),
-- agg AS (
--   SELECT
--     COALESCE(territory_id, 'ALL Territories') AS territory_id,
--     COALESCE(territory, 'All Territories') AS territory,
--     COALESCE(payer_id, 'ALL Payers') AS payer_id,
--     COALESCE(payer_name, 'All Payers') AS payer_name,
--     COUNT(DISTINCT patient_id) AS total_lives,
--     CASE
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
--     END AS rollup_level
--   FROM tagging_payer
--   GROUP BY GROUPING SETS (
--     (territory_id, territory, payer_id, payer_name),
--     (payer_id, payer_name),
--     (territory_id, territory),
--     ()
--   )
-- )
-- SELECT
--   a.*,
--   CASE
--     WHEN a.rollup_level = 'TERRITORY_PAYER' THEN
--       ROW_NUMBER() OVER (PARTITION BY a.territory_id ORDER BY a.total_lives DESC, a.payer_id, a.payer_name)
--     WHEN a.rollup_level = 'PAYER_ALL_TERRITORY' THEN
--       ROW_NUMBER() OVER (ORDER BY a.total_lives DESC, a.payer_id, a.payer_name)
--     ELSE NULL
--   END AS payer_rank
-- FROM agg a
-- ORDER BY a.rollup_level, a.total_lives DESC;

In [0]:
create or replace temporary view total_lives as 
WITH all_claims AS (
  SELECT DISTINCT * FROM (
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, KH_PLAN_ID AS plan_id, MEDICAL_EVENT_ID AS claim_id
    FROM com_edp_prd.com_raw.kom_medical_events
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id, PHARMACY_EVENT_ID AS claim_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    UNION
    SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, KH_PLAN_ID AS plan_id, MEDICAL_EVENT_ID AS claim_id
    FROM com_edp_prd.com_raw.kom_medical_events
  )
),

tagging_zip AS (
  SELECT a.*, b.PROVIDER_ZIP AS hcp_zip
  FROM all_claims a
  LEFT JOIN com_raw.kom_providers b
    ON a.npi = b.NPI AND b.PROVIDER_TYPE = 'INDIVIDUAL'
),

tagging_territory AS (
  SELECT 
    a.*, 
    COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
    COALESCE(b.territory_name, 'Unknown') AS territory
  FROM tagging_zip a
  LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
    ON TRY_CAST(a.hcp_zip AS STRING) = TRY_CAST(b.zipcode AS STRING)
),

tagging_payer AS (
  SELECT 
    a.*, 
    COALESCE(TRY_CAST(b.PAYER_ID AS STRING), 'Unknown') AS payer_id,
    COALESCE(TRY_CAST(b.PAYER_NAME AS STRING), 'Unknown') AS payer_name
  FROM tagging_territory a
  LEFT JOIN com_raw.kom_plans b
    ON a.plan_id = b.KH_PLAN_ID
),

agg AS (
  SELECT
    COALESCE(territory_id, 'ALL Territories') AS territory_id,
    COALESCE(territory, 'All Territories') AS territory,
    COALESCE(payer_id, 'ALL Payers') AS payer_id,
    COALESCE(payer_name, 'All Payers') AS payer_name,
    COUNT(DISTINCT patient_id) AS total_lives,
    CASE
      WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
      WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
      WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
      WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
    END AS rollup_level
  FROM tagging_payer
  GROUP BY GROUPING SETS (
    (territory_id, territory, payer_id, payer_name),
    (payer_id, payer_name),
    (territory_id, territory),
    ()
  )
)

SELECT *
FROM agg
ORDER BY rollup_level, total_lives DESC;

In [0]:
select * from com_raw.kom_providers where npi in ('1861429854')

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID as claim_id,
    KH_PLAN_ID as plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID as claim_id,
    coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';


CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID as claim_id,
    NDC11 AS CODE,
    KH_PLAN_ID as plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID as claim_id,
    PROCEDURE_CODE AS CODE,
    KH_PLAN_ID as plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'

UNION

SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID as claim_id,
    NDC11 AS CODE,
    coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';


CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims t ON e.PATIENT_ID = t.PATIENT_ID;

CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims
WHERE CODE IN ('54092070001', '540920700', 'J1743');

CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;

CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    claim_id,
    plan_id,
    CLAIM_SOURCE,
    TRANSACTION_STATUS
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    claim_id,
    plan_id,
    CLAIM_SOURCE,
    TRANSACTION_STATUS
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);


In [0]:
select count(distinct patient_id)
from all_patient_claims

In [0]:
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
),
hco_addition as (
  SELECT
    a.PATIENT_ID as patient_id,
    a.NPI AS hcp_npi,
    a.SPECIALTY AS hcp_specialty,
    CASE 
      WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
      ELSE b.hcp_zip 
    END AS hcp_zip,
    case
      when b.hcp_name is not null then b.hcp_name else concat(c.FIRST_NAME, " ", c.last_name) end as hcp_name,
    b.hco_npi,
    b.hco_name
FROM ranked_hcps as a
left join com_edp_prd.cmpa_insights_internal_schema.reference_file_0219 as b on a.npi = b.hcp_npi
left join com_raw.kom_providers as c on a.npi = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL'
WHERE a.HCP_RANK = 1
)
select distinct * from hco_addition

In [0]:
select * from primary_hcp

In [0]:
CREATE OR REPLACE TEMP VIEW new_patient_flags AS
SELECT
    PATIENT_ID,
    MIN(FILL_DATE) AS FIRST_EVENT_DATE,

    CASE 
        WHEN MIN(FILL_DATE) >= DATEADD(month, -1, DATE('${end_date}'))
        THEN 1 ELSE 0 
    END AS NEW_PATIENT_R1M,

    CASE 
        WHEN MIN(FILL_DATE) >= DATEADD(month, -3, DATE('${end_date}'))
        THEN 1 ELSE 0 
    END AS NEW_PATIENT_R3M

FROM all_patient_claims
GROUP BY PATIENT_ID;


In [0]:
CREATE OR REPLACE TEMP VIEW elaprase_provider_universe AS

WITH raw_provider_claims AS (

    -- ===============================
    -- MEDICAL CLAIMS (DX + NDC + PROC)
    -- ===============================
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            DIAGNOSIS_CODES LIKE '%E761%'
         OR DIAGNOSIS_CODES LIKE '%E763%'
         OR NDC11 IN ('54092070001','540920700')
         OR PROCEDURE_CODE IN (
             '99601','99602','96365','96366','J1743',
             'S9357','S9379','38206','38230','38232',
             '38240','38241','38242','38243','38250'
         )
    )
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

    UNION

    -- ===============================
    -- PHARMACY CLAIMS (DX + NDC)
    -- ===============================
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS HCP_NPI,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE (
            DIAGNOSIS_CODE IN ('E761','E763')
         OR NDC11 IN ('54092070001','540920700')
    )
    AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)

-- ===============================
-- Attach Plan + Territory Mapping
-- ===============================
SELECT DISTINCT
    r.PATIENT_ID,
    r.HCP_NPI,
    r.HCO_NPI,

    p.plan_id,
    p.PAYER_ID,
    p.PAYER_NAME,
    p.PARENT_ID,
    p.PARENT_NAME,

    p.territory_id,
    p.territory,
    p.region_id,
    p.region

FROM raw_provider_claims r
JOIN payer_master_patient_level p
    ON r.PATIENT_ID = p.patient_id;


---------- Validation
SELECT * FROM elaprase_provider_universe;


In [0]:
create or replace temporary view payer_master_patient_level as 
with eligible_patient_universe as (
  select distinct patient_id 
  from eligible_patients
),

patient_claim_counts as (
  select patient_id, count(distinct claim_id) as claims_count
  from all_patient_claims
  group by 1
),

eligible_patients_with_claims as (
  select distinct a.patient_id, b.claims_count
  from eligible_patient_universe as a 
  left join patient_claim_counts as b 
    on a.patient_id = b.patient_ids
),

patients_with_primary_hcp as (
  select a.*, b.* except(b.patient_id)
  from eligible_patients_with_claims as a
  left join primary_hcp as b 
    on a.patient_id = b.patient_id
),

patients_with_territory_region as (
  select 
    a.*, 
    b.territory_id, 
    b.territory_name as territory, 
    b.region_id, 
    b.region_name as region
  from patients_with_primary_hcp as a
  left join com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping as b 
    on a.hcp_zip = b.zipcode
),

latest_plan_per_patient as (
  select b.patient_id, b.plan_id
  from ( 
    select 
      a.patient_id, 
      a.plan_id, 
      row_number() over (
        partition by a.patient_id 
        order by a.fill_date desc, a.npi asc
      ) as rn
    from all_patient_claims as a
    where a.plan_id is not null
  ) as b
  where b.rn = 1
),

patients_with_latest_plan as (
  select a.*, b.plan_id
  from patients_with_territory_region as a
  left join latest_plan_per_patient as b 
    on a.patient_id = b.patient_id
),

patients_with_payer_attributes as (
  select distinct 
    a.*, 
    b.PAYER_ID, 
    b.PAYER_NAME, 
    b.PARENT_ID, 
    b.PARENT_NAME, 
    b.INSURANCE_SEGMENT, 
    b.INSURANCE_GROUP
  from patients_with_latest_plan as a
  left join com_edp_prd.com_raw.kom_plans as b 
    on a.plan_id = b.KH_PLAN_ID
)

-- select patient_id, claims_count, hcp_npi, hcp_specialty, hcp_zip, hcp_name, cast(coalesce(territory_id, 'Unknown') as string) as territory_id, coalesce(territory, 'Unknown') as territory, cast(coalesce(region_id, 'Unknown') as string) as region_id, coalesce(region, 'Unknown') as region, coalesce(plan_id, 'Unknown') as plan_id, coalesce(PAYER_ID, 'Unknown') as PAYER_ID, coalesce(PAYER_NAME, 'Unknown') as PAYER_NAME, coalesce(PARENT_ID, 'Unknown') as PARENT_ID, coalesce(PARENT_NAME, 'Unknown') as PARENT_NAME, coalesce(INSURANCE_SEGMENT, 'Unknown') as INSURANCE_SEGMENT, coalesce(INSURANCE_GROUP, 'Unknown')
-- from patients_with_payer_attributes;\

SELECT
  patient_id,
  claims_count,
  hcp_npi,
  hcp_specialty,
  hcp_zip,
  hcp_name,
  hco_npi,
  hco_name,

  coalesce(cast(territory_id as string), 'Unknown') as territory_id,
  COALESCE(territory, 'Unknown') AS territory,

  region_id,
  COALESCE(region, 'Unknown') AS region,

  plan_id,
  COALESCE(PAYER_ID, 'Unknown') AS PAYER_ID,
  COALESCE(PAYER_NAME, 'Unknown') AS PAYER_NAME,
  COALESCE(PARENT_ID, 'Unknown') AS PARENT_ID,
  COALESCE(PARENT_NAME, 'Unknown') AS PARENT_NAME,
  COALESCE(INSURANCE_SEGMENT, 'Unknown') AS INSURANCE_SEGMENT,
  COALESCE(INSURANCE_GROUP, 'Unknown') AS INSURANCE_GROUP
FROM patients_with_payer_attributes;

In [0]:
select * from payer_master_patient_level

### Yaman

In [0]:
-- select
--     territory_id,
--     territory as territory_name,
--     PAYER_ID,
--     PAYER_NAME,
--     PARENT_ID,
--     PARENT_NAME,

--     -- Patient counts by insurance segment
--     count(distinct case when INSURANCE_GROUP = 'MEDICARE' then patient_id end)   as MEDICARE_PATIENTS,
--     count(distinct case when INSURANCE_GROUP = 'MEDICAID' then patient_id end)   as MEDICAID_PATIENTS,
--     count(distinct case when INSURANCE_GROUP = 'COMMERCIAL' then patient_id end) as COMMERCIAL_PATIENTS,
--     count(distinct case 
--         when INSURANCE_GROUP is null 
--           or INSURANCE_GROUP not in ('MEDICARE','MEDICAID','COMMERCIAL') 
--         then patient_id end) as OTHER_PATIENTS,

--     count(distinct patient_id) as total_elaprase_patients,
--     count(distinct hcp_npi) as total_primary_hcps,
--     count(distinct hco_npi) as total_primary_hcos,
--     sum(claims_count) as total_claims

-- from payer_master_patient_level
-- group by 1,2,3,4,5,6
-- order by territory_name, payer_name;

In [0]:
-- SELECT
--   COALESCE(CAST(territory_id AS STRING), 'ALL Territories')                 AS territory_id,
--   COALESCE(territory, 'All Territories')                        AS territory_name,
--   COALESCE(CAST(payer_id AS STRING), 'ALL Payers')                     AS payer_id,
--   COALESCE(payer_name, 'All Payers')                            AS payer_name,
--   COALESCE(CAST(parent_id AS STRING), 'ALL Parents')                    AS parent_id,
--   COALESCE(parent_name, 'All Parents')                          AS parent_name,

--   COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICARE'  THEN patient_id END) AS medicare_patients,
--   COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICAID'  THEN patient_id END) AS medicaid_patients,
--   COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
--   COUNT(DISTINCT CASE
--     WHEN INSURANCE_GROUP IS NULL
--       OR INSURANCE_GROUP NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--     THEN patient_id
--   END) AS other_patients,

--   COUNT(DISTINCT patient_id) AS total_elaprase_patients,
--   COUNT(DISTINCT hcp_npi)    AS total_primary_hcps,
--   COUNT(DISTINCT hco_npi)    AS total_primary_hcos,
--   SUM(claims_count)          AS total_claims,

--   CASE
--     WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
--     WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
--     WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
--     WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
--   END AS rollup_level

-- FROM payer_master_patient_level
-- GROUP BY GROUPING SETS (
--   (territory_id, territory, payer_id, payer_name, parent_id, parent_name), -- 1) Territory + Payer
--   (payer_id, payer_name, parent_id, parent_name),                          -- 2) Payer (All Territories)
--   (territory_id, territory),                                               -- 3) Territory (All Payers)
--   ()                                                                       -- 4) National
-- )
-- ORDER BY rollup_level, territory_name, payer_name;

In [0]:
-- -- assumes your temp view `total_lives` already exists

-- WITH rollup_metrics AS (
--   SELECT
--     COALESCE(CAST(territory_id AS STRING), 'ALL Territories') AS territory_id,
--     COALESCE(territory, 'All Territories')                   AS territory_name,
--     COALESCE(CAST(payer_id AS STRING), 'ALL Payers')          AS payer_id,
--     COALESCE(payer_name, 'All Payers')                       AS payer_name,
--     COALESCE(CAST(parent_id AS STRING), 'ALL Parents')        AS parent_id,
--     COALESCE(parent_name, 'All Parents')                     AS parent_name,

--     COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
--     COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
--     COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
--     COUNT(DISTINCT CASE
--       WHEN insurance_group IS NULL
--         OR insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--       THEN patient_id
--     END) AS other_patients,

--     COUNT(DISTINCT patient_id) AS total_elaprase_patients,
--     COUNT(DISTINCT hcp_npi)    AS total_primary_hcps,
--     COUNT(DISTINCT hco_npi)    AS total_primary_hcos,
--     SUM(claims_count)          AS total_claims,

--     CASE
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
--     END AS rollup_level

--   FROM payer_master_patient_level
--   GROUP BY GROUPING SETS (
--     (territory_id, territory, payer_id, payer_name, parent_id, parent_name),
--     (payer_id, payer_name, parent_id, parent_name),
--     (territory_id, territory),
--     ()
--   )
-- )

-- SELECT
--   m.territory_id,
--   m.territory_name,
--   m.payer_id,
--   m.payer_name,
--   m.parent_id,
--   m.parent_name,
--   m.medicare_patients,
--   m.medicaid_patients,
--   m.commercial_patients,
--   m.other_patients,
--   m.total_elaprase_patients,
--   m.total_primary_hcps,
--   m.total_primary_hcos,
--   m.total_claims,
--   t.total_lives,
--   m.rollup_level
-- FROM rollup_metrics m
-- JOIN total_lives t
--   ON m.rollup_level = t.rollup_level
--  AND m.territory_id = t.territory_id
--  AND m.payer_id     = t.payer_id
-- ORDER BY m.rollup_level, m.territory_name, m.payer_name;

In [0]:
-- -- assumes your temp view `total_lives` already exists

-- WITH rollup_metrics AS (
--   SELECT
--     COALESCE(CAST(territory_id AS STRING), 'ALL Territories') AS territory_id,
--     COALESCE(territory, 'All Territories')                   AS territory_name,
--     COALESCE(CAST(payer_id AS STRING), 'ALL Payers')          AS payer_id,
--     COALESCE(payer_name, 'All Payers')                       AS payer_name,
--     COALESCE(CAST(parent_id AS STRING), 'ALL Parents')        AS parent_id,
--     COALESCE(parent_name, 'All Parents')                     AS parent_name,

--     COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
--     COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
--     COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
--     COUNT(DISTINCT CASE
--       WHEN insurance_group IS NULL
--         OR insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--       THEN patient_id
--     END) AS other_patients,

--     COUNT(DISTINCT patient_id) AS total_elaprase_patients,
--     COUNT(DISTINCT hcp_npi)    AS total_primary_hcps,
--     COUNT(DISTINCT hco_npi)    AS total_primary_hcos,
--     SUM(claims_count)          AS total_claims,

--     CASE
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
--     END AS rollup_level

--   FROM payer_master_patient_level
--   GROUP BY GROUPING SETS (
--     (territory_id, territory, payer_id, payer_name, parent_id, parent_name),
--     (payer_id, payer_name, parent_id, parent_name),
--     (territory_id, territory),
--     ()
--   )
-- ),

-- final AS (
--   SELECT
--     m.*,
--     t.total_lives
--   FROM rollup_metrics m
--   JOIN total_lives t
--     ON m.rollup_level = t.rollup_level
--    AND m.territory_id = t.territory_id
--    AND m.payer_id     = t.payer_id
-- ),

-- final_with_share AS (
--   SELECT
--     f.*,
--     -- EXACT 100% per rollup scope before rounding
--     100.0 * f.total_lives /
--       NULLIF(
--         CASE
--           WHEN f.rollup_level = 'TERRITORY_PAYER'      THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level, f.territory_id)
--           WHEN f.rollup_level = 'PAYER_ALL_TERRITORY'  THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
--           WHEN f.rollup_level = 'TERRITORY_ALL_PAYER'  THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
--           WHEN f.rollup_level = 'NATIONAL'             THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
--         END
--       ,0) AS payer_market_share_pct_unrounded
--   FROM final f
-- )

-- SELECT
--   territory_id,
--   territory_name,
--   payer_id,
--   payer_name,
--   parent_id,
--   parent_name,
--   medicare_patients,
--   medicaid_patients,
--   commercial_patients,
--   other_patients,
--   total_elaprase_patients,
--   total_primary_hcps,
--   total_primary_hcos,
--   total_claims,
--   total_lives,
--   payer_market_share_pct_unrounded AS payer_market_share_pct,
--   rollup_level
-- FROM final_with_share
-- ORDER BY rollup_level, territory_name, payer_name;

In [0]:
create or replace temporary view payer_master_final as
WITH rollup_metrics AS (
  SELECT
    COALESCE(CAST(territory_id AS STRING), 'ALL Territories') AS territory_id,
    COALESCE(territory, 'All Territories')                   AS territory_name,
    COALESCE(CAST(payer_id AS STRING), 'ALL Payers')          AS payer_id,
    COALESCE(payer_name, 'All Payers')                       AS payer_name,
    COALESCE(CAST(parent_id AS STRING), 'ALL Parents')        AS parent_id,
    COALESCE(parent_name, 'All Parents')                     AS parent_name,

    COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
    COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
    COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
    COUNT(DISTINCT CASE
      WHEN insurance_group IS NULL
        OR insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
      THEN patient_id
    END) AS other_patients,

    COUNT(DISTINCT patient_id) AS total_elaprase_patients,
    COUNT(DISTINCT hcp_npi)    AS total_primary_hcps,
    COUNT(DISTINCT hco_npi)    AS total_primary_hcos,
    SUM(claims_count)          AS total_claims,

    CASE
      WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
      WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
      WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
      WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
    END AS rollup_level

  FROM payer_master_patient_level
  GROUP BY GROUPING SETS (
    (territory_id, territory, payer_id, payer_name, parent_id, parent_name),
    (payer_id, payer_name, parent_id, parent_name),
    (territory_id, territory),
    ()
  )
),

final AS (
  SELECT
    m.*,
    t.total_lives
  FROM rollup_metrics m
  JOIN total_lives t
    ON m.rollup_level = t.rollup_level
   AND m.territory_id = t.territory_id
   AND m.payer_id     = t.payer_id
),

final_with_share AS (
  SELECT
    f.*,
    100.0 * f.total_lives /
      NULLIF(
        CASE
          WHEN f.rollup_level = 'TERRITORY_PAYER'      THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level, f.territory_id)
          WHEN f.rollup_level = 'PAYER_ALL_TERRITORY'  THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
          WHEN f.rollup_level = 'TERRITORY_ALL_PAYER'  THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
          WHEN f.rollup_level = 'NATIONAL'             THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
        END
      ,0) AS payer_market_share_pct
  FROM final f
),

final_with_rank AS (
  SELECT
    s.*,
    ROW_NUMBER() OVER (
      PARTITION BY
        CASE
          WHEN s.rollup_level = 'TERRITORY_PAYER'     THEN CONCAT(s.rollup_level, '||', s.territory_id)
          ELSE s.rollup_level
        END
      ORDER BY s.payer_market_share_pct DESC, s.total_lives DESC, s.payer_id, s.payer_name
    ) AS payer_rank
  FROM final_with_share s
)

SELECT
  territory_id,
  territory_name,
  payer_id,
  payer_name,
  parent_id,
  parent_name,
  medicare_patients,
  medicaid_patients,
  commercial_patients,
  other_patients,
  total_elaprase_patients,
  total_primary_hcps,
  total_primary_hcos,
  total_claims,
  total_lives,
  payer_market_share_pct,
  payer_rank,
  rollup_level
FROM final_with_rank
ORDER BY rollup_level, territory_name, payer_name;

In [0]:
-- select
--     '-1' as territory_id,
--     'All Territories' as territory_name,
--     PAYER_ID,
--     PAYER_NAME,
--     PARENT_ID,
--     PARENT_NAME,

--     -- Patient counts by insurance segment
--     count(distinct case when INSURANCE_GROUP = 'MEDICARE' then patient_id end)   as MEDICARE_PATIENTS,
--     count(distinct case when INSURANCE_GROUP = 'MEDICAID' then patient_id end)   as MEDICAID_PATIENTS,
--     count(distinct case when INSURANCE_GROUP = 'COMMERCIAL' then patient_id end) as COMMERCIAL_PATIENTS,
--     count(distinct case 
--         when INSURANCE_GROUP is null 
--           or INSURANCE_GROUP not in ('MEDICARE','MEDICAID','COMMERCIAL') 
--         then patient_id end) as OTHER_PATIENTS,


--     count(distinct patient_id) as total_elaprase_patients,
--     count(distinct hcp_npi) as total_hcps,
--     count(distinct hco_npi) as total_hcos

-- from payer_master_patient_level
-- group by 1,2,3,4,5,6
-- order by territory_name, payer_name;

In [0]:
-- create or replace temporary view all_patient_claims_expanded as 

-- with t1 as (select *
--   from all_patient_claims
--   where npi is not null),

-- pulling_hco_affiliations as (
--   select a.patient_id, a.npi as hcp_npi, a.fill_date, a.claim_id, a.plan_id,
--   CASE 
--       WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
--       ELSE b.hcp_zip 
--     END AS hcp_zip,
--   case
--     when b.hcp_name is not null then b.hcp_name else concat(c.FIRST_NAME, " ", c.last_name) end as hcp_name,
--   c.PRIMARY_SPECIALTY as hcp_specialty,
--   b.hco_npi, b.hco_name
--   from t1 as a
--   left join cmpa_insights_internal_schema.reference_file_0219 as b on a.npi = b.hcp_npi
--   left join com_raw.kom_providers as c on a.npi = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),
-- payer_level_info as (
--   select a.*,
--   COALESCE(b.PAYER_ID, 'Unknown') AS PAYER_ID,
--   COALESCE(b.PAYER_NAME, 'Unknown') AS PAYER_NAME,
--   COALESCE(b.PARENT_ID, 'Unknown') AS PARENT_ID,
--   COALESCE(b.PARENT_NAME, 'Unknown') AS PARENT_NAME
--   from pulling_hco_affiliations as a
--   left join com_raw.kom_plans as b on a.plan_id = b.KH_PLAN_ID
-- ),
-- territory_level_info as (
--   select a.*,
--   coalesce(cast(b.territory_id as string), 'Unknown') as territory_id,
--   COALESCE(b.territory_name, 'Unknown') AS territory_name,
--   b.region_id,
--   COALESCE(b.region_name, 'Unknown') AS region_name
--   from payer_level_info as a
--   left join cmpa_insights_internal_schema.zip_to_territory_mapping as b on a.hcp_zip = b.zipcode
-- )
-- select patient_id, claim_id, fill_date, hcp_npi, hcp_name, hcp_specialty, hco_npi, hco_name, territory_id, territory_name, region_id, region_name, payer_id, payer_name, parent_id, parent_name
-- from territory_level_info


In [0]:
-- CREATE OR REPLACE TEMPORARY VIEW all_patient_claims_expanded AS

-- WITH t1 AS (
--   SELECT *
--   FROM all_patient_claims
-- ),

-- pulling_hco_affiliations AS (
--   SELECT
--     a.patient_id,
--     a.npi            AS hcp_npi,
--     a.fill_date,
--     a.claim_id,
--     a.plan_id,

--     -- keep hcp_zip NULL when npi is NULL; otherwise pick b.hcp_zip (unless '-') then c.PROVIDER_ZIP
--     CASE
--       WHEN a.npi IS NULL THEN NULL
--       WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
--       ELSE b.hcp_zip
--     END AS hcp_zip,

--     -- hcp name / specialty / hco fields NULL when npi is NULL
--     CASE
--       WHEN a.npi IS NULL THEN NULL
--       WHEN b.hcp_name IS NOT NULL THEN b.hcp_name
--       ELSE CONCAT(c.FIRST_NAME, ' ', c.LAST_NAME)
--     END AS hcp_name,

--     CASE WHEN a.npi IS NULL THEN NULL ELSE c.PRIMARY_SPECIALTY END AS hcp_specialty,
--     CASE WHEN a.npi IS NULL THEN NULL ELSE b.hco_npi END AS hco_npi,
--     CASE WHEN a.npi IS NULL THEN NULL ELSE b.hco_name END AS hco_name

--   FROM t1 AS a
--   LEFT JOIN cmpa_insights_internal_schema.reference_file_0219 AS b
--     ON a.npi = b.hcp_npi
--   LEFT JOIN com_raw.kom_providers AS c
--     ON a.npi = c.npi AND c.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),

-- payer_level_info AS (
--   SELECT
--     a.*,
--     -- keep payer / parent NULL when hcp_npi is NULL; otherwise map and default to 'Unknown' when mapping missing
--     CASE WHEN a.hcp_npi IS NULL THEN NULL ELSE COALESCE(b.PAYER_ID, 'Unknown') END AS payer_id,
--     CASE WHEN a.hcp_npi IS NULL THEN NULL ELSE COALESCE(b.PAYER_NAME, 'Unknown') END AS payer_name,
--     CASE WHEN a.hcp_npi IS NULL THEN NULL ELSE COALESCE(b.PARENT_ID, 'Unknown') END AS parent_id,
--     CASE WHEN a.hcp_npi IS NULL THEN NULL ELSE COALESCE(b.PARENT_NAME, 'Unknown') END AS parent_name
--   FROM pulling_hco_affiliations AS a
--   LEFT JOIN com_raw.kom_plans AS b
--     ON a.plan_id = b.KH_PLAN_ID
-- ),

-- territory_level_info AS (
--   SELECT
--     a.*,
--     -- territory/region should be 'Unknown' when hcp_npi is NULL (per request),
--     -- otherwise use mapping and default to 'Unknown' when missing
--     CASE WHEN a.hcp_npi IS NULL THEN 'Unknown' ELSE COALESCE(CAST(b.territory_id AS STRING), 'Unknown') END AS territory_id,
--     CASE WHEN a.hcp_npi IS NULL THEN 'Unknown' ELSE COALESCE(b.territory_name, 'Unknown') END AS territory_name,
--     CASE WHEN a.hcp_npi IS NULL THEN 'Unknown' ELSE COALESCE(CAST(b.region_id AS STRING), 'Unknown') END AS region_id,
--     CASE WHEN a.hcp_npi IS NULL THEN 'Unknown' ELSE COALESCE(b.region_name, 'Unknown') END AS region_name
--   FROM payer_level_info AS a
--   LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping AS b
--     ON a.hcp_zip = b.zipcode
-- )

-- SELECT
--   patient_id,
--   claim_id,
--   fill_date,
--   hcp_npi,
--   hcp_name,
--   hcp_specialty,
--   hco_npi,
--   hco_name,
--   territory_id,
--   territory_name,
--   region_id,
--   region_name,
--   payer_id,
--   payer_name,
--   parent_id,
--   parent_name
-- FROM territory_level_info;

### HCP & HCO Details Tables

In [0]:
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims_expanded AS

WITH t1 AS (
  SELECT *
  FROM all_patient_claims
),

pulling_hco_affiliations AS (
  SELECT
    a.patient_id,
    a.npi            AS hcp_npi,
    a.fill_date,
    a.claim_id,
    a.plan_id,

    -- derive hcp_zip from reference file b or provider c when available;
    -- if no match (e.g. npi is null) this will be NULL
    CASE
      WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
      ELSE b.hcp_zip
    END AS hcp_zip,

    -- NULL-out HCP-specific fields when npi is NULL; otherwise prefer ref file name then provider name
    CASE
      WHEN a.npi IS NULL THEN NULL
      WHEN b.hcp_name IS NOT NULL THEN b.hcp_name
      ELSE CONCAT(c.FIRST_NAME, ' ', c.LAST_NAME)
    END AS hcp_name,

    CASE WHEN a.npi IS NULL THEN NULL ELSE c.PRIMARY_SPECIALTY END AS hcp_specialty,

    CASE WHEN a.npi IS NULL THEN NULL ELSE b.hco_npi END AS hco_npi,
    CASE WHEN a.npi IS NULL THEN NULL ELSE b.hco_name END AS hco_name

  FROM t1 AS a
  LEFT JOIN cmpa_insights_internal_schema.reference_file_0219 AS b
    ON a.npi = b.hcp_npi
  LEFT JOIN com_raw.kom_providers AS c
    ON a.npi = c.npi AND c.PROVIDER_TYPE = 'INDIVIDUAL'
),

payer_level_info AS (
  SELECT
    a.*,
    -- map payer/parent from plan_id regardless of npi presence; default to 'Unknown' if mapping missing
    COALESCE(b.PAYER_ID, 'Unknown')   AS payer_id,
    COALESCE(b.PAYER_NAME, 'Unknown') AS payer_name,
    COALESCE(b.PARENT_ID, 'Unknown')  AS parent_id,
    COALESCE(b.PARENT_NAME, 'Unknown')AS parent_name
  FROM pulling_hco_affiliations AS a
  LEFT JOIN com_raw.kom_plans AS b
    ON a.plan_id = b.KH_PLAN_ID
),

territory_level_info AS (
  SELECT
    a.*,
    -- derive territory/region from hcp_zip; if hcp_zip is NULL or mapping missing -> 'Unknown'
    COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
    COALESCE(b.territory_name, 'Unknown')               AS territory_name,
    COALESCE(CAST(b.region_id AS STRING), 'Unknown')    AS region_id,
    COALESCE(b.region_name, 'Unknown')                  AS region_name
  FROM payer_level_info AS a
  LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping AS b
    ON a.hcp_zip = b.zipcode
)

SELECT
  patient_id,
  claim_id,
  fill_date,
  hcp_npi,
  hcp_name,
  hcp_specialty,
  hco_npi,
  hco_name,
  territory_id,
  territory_name,
  region_id,
  region_name,
  payer_id,
  payer_name,
  parent_id,
  parent_name
FROM territory_level_info;

In [0]:
create or replace temporary view hcp_level_details as
SELECT
  COALESCE(territory_id, 'ALL Territories')        AS territory_id,
  COALESCE(territory_name, 'All Territories') AS territory_name,
  COALESCE(payer_id, 'ALL Payers')            AS payer_id,
  COALESCE(payer_name, 'All Payers')   AS payer_name,
  COALESCE(hcp_npi, 'ALL HCPs')             AS hcp_npi,
  COALESCE(hcp_name, 'All HCPs')       AS hcp_name,
  COALESCE(hco_npi, 'ALL HCOs')             AS hco_npi,
  COALESCE(hco_name, 'All HCOs')       AS hco_name,

  COUNT(DISTINCT patient_id) AS patient_count,
  COUNT(DISTINCT claim_id)   AS claims_count,
  MAX(fill_date)             AS last_treatment_date,

  CASE
    WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
    WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
    WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
    WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
  END AS rollup_level

FROM all_patient_claims_expanded

GROUP BY GROUPING SETS (
  -- 1) Territory + Payer + HCP/HCO (most granular)
  (territory_id, territory_name, payer_id, payer_name, hcp_npi, hcp_name, hco_npi, hco_name),

  -- 2) Payer (all territories) with HCP/HCO breakdown
  (payer_id, payer_name, hcp_npi, hcp_name, hco_npi, hco_name),

  -- 3) Territory (all payers)
  (territory_id, territory_name),

  -- 4) National
  ()
)

ORDER BY rollup_level, territory_name, payer_name, hcp_name;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW hco_level_details AS
SELECT
  COALESCE(territory_id, 'ALL Territories')    AS territory_id,
  COALESCE(territory_name, 'All Territories')  AS territory_name,
  COALESCE(payer_id, 'ALL Payers')             AS payer_id,
  COALESCE(payer_name, 'All Payers')           AS payer_name,
  COALESCE(hco_npi, 'ALL HCOs')                AS hco_npi,
  COALESCE(hco_name, 'All HCOs')               AS hco_name,

  COUNT(DISTINCT patient_id) AS patient_count,
  COUNT(DISTINCT claim_id)   AS claims_count,
  MAX(fill_date)             AS last_treatment_date,

  CASE
    WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
    WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
    WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
    WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
  END AS rollup_level

FROM all_patient_claims_expanded

GROUP BY GROUPING SETS (
  -- 1) Territory + Payer + HCO (most granular, HCP removed)
  (territory_id, territory_name, payer_id, payer_name, hco_npi, hco_name),

  -- 2) Payer (all territories) with HCO breakdown
  (payer_id, payer_name, hco_npi, hco_name),

  -- 3) Territory (all payers)
  (territory_id, territory_name),

  -- 4) National
  ()
)

ORDER BY rollup_level, territory_name, payer_name, hco_name;

In [0]:
select * from hco_level_details 

# Pooja

In [0]:
CREATE OR REPLACE TEMP VIEW patient_current_age AS
SELECT distinct
    p.patient_id,
    YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS CURRENT_AGE
FROM payer_master_patient_level p
LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics d
    ON p.patient_id = d.PATIENT_ID
WHERE d.PATIENT_YOB IS NOT NULL;


In [0]:
CREATE OR REPLACE TEMP VIEW patient_claim_metrics_with_rates AS
SELECT distinct
    patient_id,

    -- All claims (DX + RX)
    COUNT(DISTINCT claim_id) AS TOTAL_CLAIMS,

    -- Pharmacy-only denominator
    COUNT(DISTINCT CASE
        WHEN CLAIM_SOURCE = 'PHARMACY'
        THEN claim_id
    END) AS PHARMACY_TOTAL_CLAIMS,

    COUNT(DISTINCT CASE
        WHEN CLAIM_SOURCE = 'PHARMACY'
         AND UPPER(TRANSACTION_STATUS) = 'PAID'
        THEN claim_id
    END) AS APPROVED_FILLS,

    COUNT(DISTINCT CASE
        WHEN CLAIM_SOURCE = 'PHARMACY'
         AND UPPER(TRANSACTION_STATUS) = 'REJECTED'
        THEN claim_id
    END) AS REJECTED_FILLS,

    COUNT(DISTINCT CASE
        WHEN CLAIM_SOURCE = 'PHARMACY'
         AND UPPER(TRANSACTION_STATUS) = 'REVERSED'
        THEN claim_id
    END) AS REVERSED_FILLS,

    -- ✅ Correct rates (Pharmacy denominator)
    ROUND(
        CASE
            WHEN COUNT(DISTINCT CASE WHEN CLAIM_SOURCE = 'PHARMACY' THEN claim_id END) = 0
            THEN 0
            ELSE 100.0 *
                 COUNT(DISTINCT CASE
                     WHEN CLAIM_SOURCE = 'PHARMACY'
                      AND UPPER(TRANSACTION_STATUS) = 'PAID'
                     THEN claim_id
                 END)
                 /
                 COUNT(DISTINCT CASE
                     WHEN CLAIM_SOURCE = 'PHARMACY'
                     THEN claim_id
                 END)
        END
    , 2) AS ELAPRASE_APPROVAL_RATE,

    ROUND(
        CASE
            WHEN COUNT(DISTINCT CASE WHEN CLAIM_SOURCE = 'PHARMACY' THEN claim_id END) = 0
            THEN 0
            ELSE 100.0 *
                 COUNT(DISTINCT CASE
                     WHEN CLAIM_SOURCE = 'PHARMACY'
                      AND UPPER(TRANSACTION_STATUS) = 'REJECTED'
                     THEN claim_id
                 END)
                 /
                 COUNT(DISTINCT CASE
                     WHEN CLAIM_SOURCE = 'PHARMACY'
                     THEN claim_id
                 END)
        END
    , 2) AS ELAPRASE_REJECTION_RATE,

    ROUND(
        CASE
            WHEN COUNT(DISTINCT CASE WHEN CLAIM_SOURCE = 'PHARMACY' THEN claim_id END) = 0
            THEN 0
            ELSE 100.0 *
                 COUNT(DISTINCT CASE
                     WHEN CLAIM_SOURCE = 'PHARMACY'
                      AND UPPER(TRANSACTION_STATUS) = 'REVERSED'
                     THEN claim_id
                 END)
                 /
                 COUNT(DISTINCT CASE
                     WHEN CLAIM_SOURCE = 'PHARMACY'
                     THEN claim_id
                 END)
        END
    , 2) AS ELAPRASE_REVERSED_RATE

FROM all_patient_claims
GROUP BY patient_id;


In [0]:
CREATE OR REPLACE TEMP VIEW payer_master_patient_level_extended AS
SELECT distinct
    p.*,

    -- New Patient Flags
    COALESCE(n.NEW_PATIENT_R1M, 0)  AS NEW_PATIENT_R1M,
    COALESCE(n.NEW_PATIENT_R3M, 0)  AS NEW_PATIENT_R3M,
    n.FIRST_EVENT_DATE,

    -- Claim Metrics
    COALESCE(c.TOTAL_CLAIMS, 0)           AS TOTAL_CLAIMS,
    COALESCE(c.PHARMACY_TOTAL_CLAIMS, 0)  AS PHARMACY_TOTAL_CLAIMS,
    COALESCE(c.APPROVED_FILLS, 0)         AS APPROVED_FILLS,
    COALESCE(c.REJECTED_FILLS, 0)         AS REJECTED_FILLS,
    COALESCE(c.REVERSED_FILLS, 0)         AS REVERSED_FILLS,
    COALESCE(c.ELAPRASE_APPROVAL_RATE, 0) AS ELAPRASE_APPROVAL_RATE,
    COALESCE(c.ELAPRASE_REJECTION_RATE, 0) AS ELAPRASE_REJECTION_RATE,
    COALESCE(c.ELAPRASE_REVERSED_RATE, 0)  AS ELAPRASE_REVERSED_RATE,

    -- Age
    a.CURRENT_AGE,
    CASE WHEN a.CURRENT_AGE < 5 THEN 1 ELSE 0 END AS AGE_LT_5_YRS,
    CASE WHEN a.CURRENT_AGE BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS AGE_5_TO_10_YRS,
    CASE WHEN a.CURRENT_AGE BETWEEN 11 AND 18 THEN 1 ELSE 0 END AS AGE_11_TO_18_YRS,
    CASE WHEN a.CURRENT_AGE > 18 THEN 1 ELSE 0 END AS AGE_GT_18_YRS,

    -- Engagement (Direct Join – No Mapping Table)
    COALESCE(i.PIE_COMPLETED, 'NO') AS PIE_COMPLETED,
    COALESCE(i.ACCOUNT_DIRECTOR, '-') AS ACCOUNT_DIRECTOR

FROM payer_master_patient_level p

LEFT JOIN new_patient_flags n
    ON p.patient_id = n.patient_id

LEFT JOIN patient_claim_metrics_with_rates c
    ON p.patient_id = c.patient_id

LEFT JOIN patient_current_age a
    ON p.patient_id = a.patient_id

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
    ON UPPER(p.PAYER_NAME) = UPPER(i.PAYER_ACCOUNT_NAME);

---------- Validation
SELECT distinct * FROM payer_master_patient_level_extended;


In [0]:
CREATE OR REPLACE TEMP VIEW payer_territory_patient_summary AS
SELECT DISTINCT
    territory_id,
    territory AS territory_name,
    region_id,
    region AS region_name,

    PAYER_ID,
    PAYER_NAME,
    PARENT_ID,
    PARENT_NAME,

    -- Patient Counts
    COUNT(DISTINCT patient_id) AS TOTAL_PATIENTS,

    -- Insurance Split
    COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICARE' THEN patient_id END)   AS MEDICARE_PATIENTS,
    COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICAID' THEN patient_id END)   AS MEDICAID_PATIENTS,
    COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'COMMERCIAL' THEN patient_id END) AS COMMERCIAL_PATIENTS,
    COUNT(DISTINCT CASE
        WHEN INSURANCE_GROUP IS NULL
          OR INSURANCE_GROUP NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
        THEN patient_id END) AS OTHER_PATIENTS,

    -- New Patients
    SUM(NEW_PATIENT_R1M) AS NEW_PATIENTS_R1M,
    SUM(NEW_PATIENT_R3M) AS NEW_PATIENTS_R3M,

    -- Age Buckets
    SUM(AGE_LT_5_YRS)     AS AGE_LT_5_YRS,
    SUM(AGE_5_TO_10_YRS)  AS AGE_5_TO_10_YRS,
    SUM(AGE_11_TO_18_YRS) AS AGE_11_TO_18_YRS,
    SUM(AGE_GT_18_YRS)    AS AGE_GT_18_YRS,

    -- Provider Counts
    COUNT(DISTINCT hcp_npi) AS TOTAL_primary_HCPS,
    COUNT(DISTINCT hco_npi) AS TOTAL_primary_HCOS

FROM payer_master_patient_level_extended
GROUP BY
    territory_id,
    territory,
    region_id,
    region,
    PAYER_ID,
    PAYER_NAME,
    PARENT_ID,
    PARENT_NAME;


  ---------- Validation
SELECT distinct * FROM payer_territory_patient_summary;



In [0]:
CREATE OR REPLACE TEMP VIEW payer_master_patient_level_test AS

WITH eligible_patient_universe AS (
  SELECT DISTINCT patient_id 
  FROM eligible_patients
),

patient_claim_counts AS (
  SELECT DISTINCT patient_id, COUNT(DISTINCT claim_id) AS claims_count
  FROM all_patient_claims
  GROUP BY 1
),

eligible_patients_with_claims AS (
  SELECT DISTINCT a.patient_id, b.claims_count
  FROM eligible_patient_universe a 
  LEFT JOIN patient_claim_counts b 
    ON a.patient_id = b.patient_id
),

patients_with_primary_hcp AS (
  SELECT DISTINCT a.*, b.* EXCEPT(b.patient_id)
  FROM eligible_patients_with_claims a
  LEFT JOIN primary_hcp b 
    ON a.patient_id = b.patient_id
),

patients_with_territory_region AS (
  SELECT DISTINCT
    a.*, 
    b.territory_id, 
    b.territory_name AS territory, 
    b.region_id, 
    b.region_name AS region
  FROM patients_with_primary_hcp a
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
    ON a.hcp_zip = b.zipcode
),

latest_plan_per_patient AS (
  SELECT patient_id, plan_id
  FROM (
    SELECT DISTINCT
      patient_id, 
      plan_id,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id 
        ORDER BY fill_date DESC, npi ASC
      ) AS rn
    FROM all_patient_claims
    WHERE plan_id IS NOT NULL
  )
  WHERE rn = 1
),

patients_with_latest_plan AS (
  SELECT a.*, b.plan_id
  FROM patients_with_territory_region a
  LEFT JOIN latest_plan_per_patient b
    ON a.patient_id = b.patient_id
),

patients_with_payer_attributes AS (
  SELECT DISTINCT  
    a.*, 
    b.PAYER_ID, 
    b.PAYER_NAME, 
    b.PARENT_ID, 
    b.PARENT_NAME, 
    b.INSURANCE_SEGMENT, 
    b.INSURANCE_GROUP
  FROM patients_with_latest_plan a
  LEFT JOIN com_edp_prd.com_raw.kom_plans b
    ON a.plan_id = b.KH_PLAN_ID
),

-- =========================
-- NEW PATIENT FLAGS
-- =========================
new_patient_flags AS (
  SELECT DISTINCT
    patient_id,
    MIN(fill_date) AS FIRST_EVENT_DATE,
    CASE WHEN MIN(fill_date) >= DATEADD(month, -1, DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R1M,
    CASE WHEN MIN(fill_date) >= DATEADD(month, -3, DATE('${end_date}')) THEN 1 ELSE 0 END AS NEW_PATIENT_R3M
  FROM all_patient_claims
  GROUP BY patient_id
),

-- =========================
-- CLAIM METRICS
-- =========================
patient_claim_metrics AS (
  SELECT DISTINCT
    patient_id,
    COUNT(DISTINCT claim_id) AS TOTAL_CLAIMS,
    COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' THEN claim_id END) AS PHARMACY_TOTAL_CLAIMS,
    COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' AND UPPER(TRANSACTION_STATUS)='PAID' THEN claim_id END) AS APPROVED_FILLS,
    COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' AND UPPER(TRANSACTION_STATUS)='REJECTED' THEN claim_id END) AS REJECTED_FILLS,
    COUNT(DISTINCT CASE WHEN CLAIM_SOURCE='PHARMACY' AND UPPER(TRANSACTION_STATUS)='REVERSED' THEN claim_id END) AS REVERSED_FILLS
  FROM all_patient_claims
  GROUP BY patient_id
),

-- =========================
-- AGE
-- =========================
patient_age AS (
  SELECT
    p.patient_id,
    YEAR(CURRENT_DATE) - YEAR(d.PATIENT_YOB) AS CURRENT_AGE
  FROM patients_with_payer_attributes p
  LEFT JOIN com_edp_prd.com_raw.kom_patient_demographics d
    ON p.patient_id = d.PATIENT_ID
)

-- =========================
-- FINAL SELECT
-- =========================
SELECT DISTINCT
  p.*,

  -- New Patient
  COALESCE(n.NEW_PATIENT_R1M,0) AS NEW_PATIENT_R1M,
  COALESCE(n.NEW_PATIENT_R3M,0) AS NEW_PATIENT_R3M,
  n.FIRST_EVENT_DATE,

  -- Claims
  COALESCE(c.TOTAL_CLAIMS,0) AS TOTAL_CLAIMS,
  COALESCE(c.PHARMACY_TOTAL_CLAIMS,0) AS PHARMACY_TOTAL_CLAIMS,
  COALESCE(c.APPROVED_FILLS,0) AS APPROVED_FILLS,
  COALESCE(c.REJECTED_FILLS,0) AS REJECTED_FILLS,
  COALESCE(c.REVERSED_FILLS,0) AS REVERSED_FILLS,

  -- Rates
  CASE WHEN c.PHARMACY_TOTAL_CLAIMS=0 THEN 0
       ELSE ROUND(100.0*c.APPROVED_FILLS/c.PHARMACY_TOTAL_CLAIMS,2)
  END AS ELAPRASE_APPROVAL_RATE,

  CASE WHEN c.PHARMACY_TOTAL_CLAIMS=0 THEN 0
       ELSE ROUND(100.0*c.REJECTED_FILLS/c.PHARMACY_TOTAL_CLAIMS,2)
  END AS ELAPRASE_REJECTION_RATE,

  CASE WHEN c.PHARMACY_TOTAL_CLAIMS=0 THEN 0
       ELSE ROUND(100.0*c.REVERSED_FILLS/c.PHARMACY_TOTAL_CLAIMS,2)
  END AS ELAPRASE_REVERSED_RATE,

  -- Age
  a.CURRENT_AGE,
  CASE WHEN a.CURRENT_AGE < 5 THEN 1 ELSE 0 END AS AGE_LT_5_YRS,
  CASE WHEN a.CURRENT_AGE BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS AGE_5_TO_10_YRS,
  CASE WHEN a.CURRENT_AGE BETWEEN 11 AND 18 THEN 1 ELSE 0 END AS AGE_11_TO_18_YRS,
  CASE WHEN a.CURRENT_AGE > 18 THEN 1 ELSE 0 END AS AGE_GT_18_YRS,

  -- Engagement
  COALESCE(i.PIE_COMPLETED,'NO') AS PIE_COMPLETED,
  COALESCE(i.ACCOUNT_DIRECTOR,'-') AS ACCOUNT_DIRECTOR

FROM patients_with_payer_attributes p
LEFT JOIN new_patient_flags n ON p.patient_id=n.patient_id
LEFT JOIN patient_claim_metrics c ON p.patient_id=c.patient_id
LEFT JOIN patient_age a ON p.patient_id=a.patient_id
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_info i
  ON UPPER(p.PAYER_NAME)=UPPER(i.PAYER_ACCOUNT_NAME);

  ------ VALIDATION
  SELECT DISTINCT * FROM payer_master_patient_level_test;


In [0]:
-- CREATE OR REPLACE TEMP VIEW national_summary AS
SELECT DISTINCT

    '-1' AS territory_id,
    'NATIONAL_ROLLUP' AS territory_name,

    '-1' AS region_id,
    'NATIONAL_ROLLUP' AS region_name,

    'NATIONAL_ROLLUP' AS PAYER_ID,
    'NATIONAL_ROLLUP' AS PAYER_NAME,
    'NATIONAL_ROLLUP' AS PARENT_ID,
    'NATIONAL_ROLLUP' AS PARENT_NAME,

    -- =====================
    -- PATIENT COUNTS
    -- =====================
    COUNT(DISTINCT patient_id) AS TOTAL_PATIENTS,

    COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICARE' THEN patient_id END)   AS MEDICARE_PATIENTS,
    COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'MEDICAID' THEN patient_id END)   AS MEDICAID_PATIENTS,
    COUNT(DISTINCT CASE WHEN INSURANCE_GROUP = 'COMMERCIAL' THEN patient_id END) AS COMMERCIAL_PATIENTS,
    COUNT(DISTINCT CASE 
        WHEN INSURANCE_GROUP IS NULL 
          OR INSURANCE_GROUP NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
        THEN patient_id 
    END) AS OTHER_PATIENTS,

    -- =====================
    -- NEW PATIENTS
    -- =====================
    SUM(NEW_PATIENT_R1M) AS NEW_PATIENTS_R1M,
    SUM(NEW_PATIENT_R3M) AS NEW_PATIENTS_R3M,

    -- =====================
    -- AGE BUCKETS
    -- =====================
    SUM(AGE_LT_5_YRS)     AS AGE_LT_5_YRS,
    SUM(AGE_5_TO_10_YRS)  AS AGE_5_TO_10_YRS,
    SUM(AGE_11_TO_18_YRS) AS AGE_11_TO_18_YRS,
    SUM(AGE_GT_18_YRS)    AS AGE_GT_18_YRS,

    -- =====================
    -- CLAIM METRICS
    -- =====================
    SUM(TOTAL_CLAIMS) AS TOTAL_CLAIMS,
    SUM(PHARMACY_TOTAL_CLAIMS) AS PHARMACY_TOTAL_CLAIMS,
    SUM(APPROVED_FILLS) AS APPROVED_FILLS,
    SUM(REJECTED_FILLS) AS REJECTED_FILLS,
    SUM(REVERSED_FILLS) AS REVERSED_FILLS,

    -- =====================
    -- RATES (Correct Denominator)
    -- =====================
    CASE 
        WHEN SUM(PHARMACY_TOTAL_CLAIMS) = 0 THEN 0
        ELSE ROUND(100.0 * SUM(APPROVED_FILLS) / SUM(PHARMACY_TOTAL_CLAIMS), 2)
    END AS ELAPRASE_APPROVAL_RATE,

    CASE 
        WHEN SUM(PHARMACY_TOTAL_CLAIMS) = 0 THEN 0
        ELSE ROUND(100.0 * SUM(REJECTED_FILLS) / SUM(PHARMACY_TOTAL_CLAIMS), 2)
    END AS ELAPRASE_REJECTION_RATE,

    CASE 
        WHEN SUM(PHARMACY_TOTAL_CLAIMS) = 0 THEN 0
        ELSE ROUND(100.0 * SUM(REVERSED_FILLS) / SUM(PHARMACY_TOTAL_CLAIMS), 2)
    END AS ELAPRASE_REVERSED_RATE,

    -- =====================
    -- PROVIDER COUNTS
    -- =====================
    COUNT(DISTINCT hcp_npi) AS TOTAL_primary_HCPS,
    COUNT(DISTINCT hco_npi) AS TOTAL_primary_HCOS,

    "-" AS PIE_COMPLETED,
    "-" AS ACCOUNT_DIRECTOR

FROM payer_master_patient_level_TEST;
